# Notebook 06 — Cross-Ecosystem Lake Experiment
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polar-bear-after-lunch/AareML/blob/main/notebooks/06_cross_ecosystem_lake.ipynb)

Applies AareML baselines to LakeBeD-US Lake Mendota surface data. Quantifies the 3.4× river vs lake DO predictability gap and shows Ridge regression outperforms the published LakeBeD-US LSTM.

# Notebook 06 — Cross-Ecosystem Lake Experiment
## Applying AareML Baselines to LakeBeD-US Lake Mendota

**Goal:** Apply the same baseline models (Persistence, Climatology, Ridge) used on Swiss
rivers (CAMELS-CH-Chem) to US lake data (LakeBeD-US, Lake Mendota) and quantify the
river vs. lake predictability gap.

**Dataset:** LakeBeD-US Computer Science Edition (McAfee et al. 2025), Lake Mendota (ME),
high-frequency surface buoy data (depth ≈ 1m), daily medians 2006–2023.

**Key finding:** Lake Ridge DO RMSE = 1.030 mg/L — 3.4× higher than Swiss river Ridge
(0.303 mg/L), confirming rivers are substantially more predictable than lakes.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

# AareML modules
from src.config import LOOKBACK, HORIZON
from src.metrics import metrics_table

FIGURES_DIR = Path("../figures")
RESULTS_DIR = Path("../results")
DATA_DIR    = Path("../data/lakebed-us")

## 1. Load and Explore Lake Mendota Data

In [ ]:
# Load pre-processed daily surface data
# (original: 101M high-frequency rows processed to daily medians in pre-processing step)
lake = pd.read_csv(DATA_DIR / "ME_daily_surface.csv",
                    parse_dates=["date"], index_col="date")

# Reindex to continuous daily frequency
idx = pd.date_range(lake.index.min(), lake.index.max(), freq="D")
lake = lake.reindex(idx)

print(f"Lake Mendota surface data: {len(lake)} days")
print(f"Date range: {lake.index.min().date()} \u2192 {lake.index.max().date()}")
print(f"\nCoverage:")
for col in lake.columns:
    cov = lake[col].notna().mean()
    rng = f"[{lake[col].min():.2f}, {lake[col].max():.2f}]"
    print(f"  {col:12s}: {cov:.1%}  range {rng}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(lake.index, lake["do"], color="#01696F", linewidth=0.8, alpha=0.8)
axes[0].set_ylabel("DO (mg/L)")
axes[0].set_title("Lake Mendota \u2014 Surface DO (1m depth, daily median)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(lake.index, lake["temp"], color="#A84B2F", linewidth=0.8, alpha=0.8)
axes[1].set_ylabel("Temperature (\u00b0C)")
axes[1].set_title("Lake Mendota \u2014 Surface Temperature")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_mendota_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 06_mendota_timeseries.png")

## 2. Data Preparation and Splits

Following the LakeBeD-US benchmark protocol: chronological 80/10/10 split, 
21-day lookback → 14-day forecast horizon, surface variables as features.

**Variable mapping:**
| LakeBeD-US | AareML River | Note |
|---|---|---|
| do (mg/L) | O2C_sensor | Primary target |
| temp (°C) | temp_sensor | Secondary target |
| chla_rfu | — | Chlorophyll a (lake-specific) |
| phyco | — | Phycocyanin (lake-specific) |
| par | — | Photosynthetically active radiation |

In [ ]:
LAKE_FEATURES = ["do", "temp", "chla_rfu", "phyco"]
LAKE_TARGETS  = ["do", "temp"]
N_FEAT = len(LAKE_FEATURES)
N_TGT  = len(LAKE_TARGETS)

# Focus on post-2006 (per LakeBeD-US benchmark)
lake_use = lake[lake.index >= "2006-01-01"][LAKE_FEATURES].copy()

# Chronological 80/10/10 split
n = len(lake_use)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)
train = lake_use.iloc[:n_train]
val   = lake_use.iloc[n_train:n_train+n_val]
test  = lake_use.iloc[n_train+n_val:]

print(f"Train: {len(train)} days ({train.index.min().date()} \u2192 {train.index.max().date()})")
print(f"Val:   {len(val)}   days ({val.index.min().date()} \u2192 {val.index.max().date()})")
print(f"Test:  {len(test)}  days ({test.index.min().date()} \u2192 {test.index.max().date()})")

# Impute with training means (same protocol as AareML)
train_means = train.mean()
print(f"\nTraining means: {train_means.round(2).to_dict()}")

def make_lake_windows(df, means, lookback=LOOKBACK, horizon=HORIZON):
    """Build sliding windows, skip any with NaN in target horizon."""
    data = df.fillna(means).values.astype(np.float32)
    tgt_idx = [df.columns.get_loc(c) for c in LAKE_TARGETS]
    X_list, y_list, dates = [], [], []
    for i in range(lookback, len(data) - horizon + 1):
        y_block = df.iloc[i:i+horizon][LAKE_TARGETS].values
        if np.isnan(y_block).any():
            continue
        X_list.append(data[i-lookback:i])
        y_list.append(y_block.astype(np.float32))
        dates.append(df.index[i])
    if not X_list:
        raise ValueError("No valid windows found")
    return np.array(X_list), np.array(y_list), dates

X_train, y_train, _ = make_lake_windows(train, train_means)
X_val,   y_val,   _ = make_lake_windows(val,   train_means)
X_test,  y_test,  d_test = make_lake_windows(test, train_means)

print(f"\nWindows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

# Scale features
feat_scaler = StandardScaler().fit(X_train.reshape(-1, N_FEAT))

## 3. Baseline Models

In [ ]:
from tqdm.auto import tqdm
# Flatten inputs for Ridge
X_tr_flat = feat_scaler.transform(X_train.reshape(-1, N_FEAT)).reshape(len(X_train), -1)
X_va_flat = feat_scaler.transform(X_val.reshape(-1, N_FEAT)).reshape(len(X_val), -1)
X_te_flat = feat_scaler.transform(X_test.reshape(-1, N_FEAT)).reshape(len(X_test), -1)
X_trv = np.vstack([X_tr_flat, X_va_flat])
y_trv = np.vstack([y_train, y_val])

# ── Ridge ──────────────────────────────────────────────────────────────────────────
print("Training Ridge (28 models: 2 targets \u00d7 14 horizons)...")
alphas = np.logspace(-3, 3, 20)
y_pred_ridge = np.zeros_like(y_test)
for t in tqdm(range(N_TGT), desc="Ridge targets"):
    for h in range(HORIZON):
        m = RidgeCV(alphas=alphas).fit(X_trv, y_trv[:, h, t])
        y_pred_ridge[:, h, t] = m.predict(X_te_flat)
print("  Ridge: done")

# ── Persistence ──────────────────────────────────────────────────────────────────
y_pred_persist = np.zeros_like(y_test)
for t in range(N_TGT):
    last_obs = X_test[:, -1, t]
    for h in range(HORIZON):
        y_pred_persist[:, h, t] = last_obs

# ── Climatology ──────────────────────────────────────────────────────────────────
train_filled = train.fillna(train_means)
doy_lookup = {}
for t, tgt in enumerate(LAKE_TARGETS):
    doy_lookup[t] = train_filled.groupby(train_filled.index.dayofyear)[tgt].median()

y_pred_clim = np.zeros_like(y_test)
for i, base_date in enumerate(d_test):
    for h in range(HORIZON):
        fcast_doy = (base_date + pd.Timedelta(days=h+1)).dayofyear
        for t in range(N_TGT):
            y_pred_clim[i, h, t] = doy_lookup[t].get(fcast_doy, train_means[LAKE_TARGETS[t]])

print("All baselines computed.")

In [ ]:
def rmse(yt, yp): return float(np.sqrt(np.mean((yt - yp)**2)))
def mae(yt, yp):  return float(np.mean(np.abs(yt - yp)))
def nse(yt, yp):  return float(1 - np.sum((yt-yp)**2) / np.sum((yt-yt.mean())**2))

rows = []
for name, yp in [("Persistence", y_pred_persist),
                  ("Climatology",  y_pred_clim),
                  ("Ridge",        y_pred_ridge)]:
    for t, (tgt, unit) in enumerate(zip(LAKE_TARGETS, ["mg/L", "\u00b0C"])):
        rows.append({
            "Model":  name,
            "Target": f"{tgt.upper()} ({unit})",
            "RMSE":   round(rmse(y_test[:,:,t], yp[:,:,t]), 3),
            "MAE":    round(mae(y_test[:,:,t],  yp[:,:,t]), 3),
            "NSE":    round(nse(y_test[:,:,t].ravel(), yp[:,:,t].ravel()), 3),
        })

# Add LakeBeD-US LSTM reference
rows.append({"Model": "LakeBeD-US LSTM (ref.)", "Target": "DO (mg/L)",
             "RMSE": 1.400, "MAE": float("nan"), "NSE": float("nan")})

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))
results_df.to_csv(RESULTS_DIR / "lake_mendota_results.csv", index=False)

## 4. Results Visualisation

In [ ]:
# ── Per-horizon RMSE ───────────────────────────────────────────────────────────────────
colors = {"Persistence": "#DA7101", "Climatology": "#006494", "Ridge": "#01696F"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (t, unit) in zip(axes, [(0, "mg/L"), (1, "\u00b0C")]):
    tgt = LAKE_TARGETS[t]
    for name, yp in [("Persistence", y_pred_persist),
                      ("Climatology", y_pred_clim),
                      ("Ridge",       y_pred_ridge)]:
        rmse_h = [rmse(y_test[:,h,t], yp[:,h,t]) for h in range(HORIZON)]
        ax.plot(range(1, HORIZON+1), rmse_h, "o-", color=colors[name],
                linewidth=2, markersize=4, label=name)
    ax.set_xlabel("Forecast horizon (days)")
    ax.set_ylabel(f"RMSE ({unit})")
    ax.set_title(f"{tgt.upper()} \u2014 Lake Mendota baselines")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("Lake Mendota \u2014 Baseline RMSE by Forecast Horizon", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_lake_mendota_rmse_by_horizon.png", dpi=150, bbox_inches="tight")
plt.show()

# ── River vs Lake comparison ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
river_rmse = {"do": 0.303, "temp": 1.261}
lake_lstm_do = 1.40

for ax, (t, tgt, unit) in zip(axes, [(0,"do","mg/L"),(1,"temp","\u00b0C")]):
    lake_r = [rmse(y_test[:,h,t], y_pred_ridge[:,h,t]) for h in range(HORIZON)]
    ax.plot(range(1, HORIZON+1), lake_r, "o-", color="#A84B2F",
            linewidth=2, markersize=4, label="Lake Mendota (Ridge)")
    ax.axhline(river_rmse[tgt], color="#01696F", linewidth=2, linestyle="--",
               label=f"River Gauge 2473 (Ridge)")
    if tgt == "do":
        ax.axhline(lake_lstm_do, color="#7A39BB", linewidth=1.5, linestyle=":",
                   label="LakeBeD-US LSTM (McAfee et al. 2025)")
    ax.set_xlabel("Forecast horizon (days)")
    ax.set_ylabel(f"RMSE ({unit})")
    ax.set_title(f"{tgt.upper()} \u2014 Swiss River vs US Lake")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle("Cross-Ecosystem Comparison: Swiss Rivers vs US Lake Mendota",
             fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_river_vs_lake_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figures saved.")

## 5. Cross-Ecosystem Interpretation

### Key finding
The Ridge baseline on Lake Mendota achieves **DO RMSE = 1.030 mg/L** — 
**3.4× higher** than the Swiss river Ridge (0.303 mg/L). This confirms that Swiss
Alpine rivers are substantially more predictable than US temperate lakes under 
the same task formulation.

Remarkably, the AareML Ridge baseline on lake data (**1.030 mg/L**) already beats 
the published LakeBeD-US LSTM (**1.40 mg/L**), suggesting Ridge regression is 
a surprisingly strong baseline for lake DO when multi-week sensor history is available.

### Why are rivers more predictable?
| Factor | River | Lake |
|--------|-------|------|
| DO autocorrelation | High — stable reaeration | Low — stratification disrupts |
| Seasonal forcing | Strong Alpine cycle | Moderate, disrupted by mixing events |
| Horizon sensitivity | Slow degradation | Rapid degradation after day 3–5 |
| Missing data | 97% coverage (gauge 2473) | 51% coverage (Mendota DO) |

### Limitations of this comparison
- River model was trained on Swiss Alpine conditions; lake model uses same protocol
- Lake Mendota has only 51% DO coverage vs 97% for gauge 2473
- No LSTM has been run on lake data yet — Ridge vs LSTM comparison pending

In [ ]:
print("="*70)
print("CROSS-ECOSYSTEM SUMMARY")
print("="*70)
print(f"\n{'Model':25s} {'DO RMSE':>10s}  {'Temp RMSE':>10s}")
print("-"*50)
for _, row in results_df.iterrows():
    if "DO" in row["Target"]:
        do_rmse = row["RMSE"]
        temp_row = results_df[(results_df["Model"]==row["Model"]) & 
                              (results_df["Target"].str.contains("Temp|TEMP"))]
        temp_rmse = temp_row["RMSE"].values[0] if len(temp_row) > 0 else float("nan")
        temp_str = f"{temp_rmse:6.3f}" if not np.isnan(temp_rmse) else "   —   "
        print(f"  {row['Model']:23s} {do_rmse:>8.3f} mg/L  {temp_str}")

print(f"\nRiver/Lake RMSE ratio:")
print(f"  DO:   0.303 / 1.030 = {0.303/1.030:.2f}\u00d7 (river {0.303/1.030*100:.0f}% of lake error)")
print(f"  Temp: 1.261 / 2.244 = {1.261/2.244:.2f}\u00d7 (river {1.261/2.244*100:.0f}% of lake error)")